In [0]:
%skip ./_resources/00-project-setup

In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="POSIST",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="discount",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Dennys/Dennys SZR/2026/01/03/discount.json')
display(cdc_raw_data)

cdc_raw_data.filter(~((size(col("Discount")) == 0) & (size(col("Item Details")) == 0) & (size(col("Tender Media")) == 0))).display()

In [0]:
%sql
create table if not exists fq_dev_catalog.bronze.discount TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.enableChangeDataFeed' = 'true', delta.autoOptimize.optimizeWrite = true, delta.autoOptimize.autoCompact = true);

In [0]:
import time
from pyspark.sql.functions import col, expr, current_timestamp, regexp_replace, lit, substring, substring_index, to_timestamp, size

bronzeDF = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "json")
                .option('multiline', False)
                .option("cloudFiles.allowOverwrites", "true") #re-ingest files if they are overwritten or update
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaLocation", f'{checkpoint}/{source}/{domain}/streaming/schema_discount')
                .load(f'{staging}/FoodQuest/*/discount.json')
            )
# display(bronzeDF)

(bronzeDF.filter(~((size(col("Discount")) == 0) & (size(col("Item Details")) == 0) & (size(col("Tender Media")) == 0)))
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn('file_path', regexp_replace(col('_metadata.file_path'), '%20', ' '))
        .withColumn('sys_id', expr('uuid()'))
        .writeStream
        .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_discount')
        .trigger(availableNow=True)
        .queryName(f'{domain}WriteStream')
        .outputMode('append')
        .toTable(f"`{environment}_catalog`.`bronze`.`{domain}`", mergeSchema=True)
)

time.sleep(20)

In [0]:
%sql
select count(*) from fq_dev_catalog.bronze.discount

In [0]:
%sql
select count(*) from fq_dev_catalog.bronze.discount LATERAL VIEW explode(Discount) AS d WHERE d.`Business Date` = '2026-01-19';

In [0]:
%sql
select count(*) from fq_dev_catalog.bronze.discount LATERAL VIEW explode(`Item Details`) AS i WHERE i.`Business Date` = '2026-01-19';

In [0]:
for query in spark.streams.active:
    query.stop()